# Neuroimaging stress workshop — facilitator notebook

This is the answer key, run sheet, science-language guide, and technical check for `NeuroPET_exercise.ipynb`. Run all cells before the session.

> MRI anatomy and atlas labels are open template resources. PET and fMRI signals are deterministic educational simulations, not patient data or validated stress biomarkers.

## RUN THIS — pre-flight check

In [ ]:
import platform
import matplotlib
import numpy as np
from workshop_helpers import (
    calculate_correlation_map, compare_pet_patterns, how_well_matched,
    load_workshop_data, measure_region, normalise_match_score, optimise,
    shift_image, show_alignment, show_fmri_result, show_region_atlas,
    show_region_measurements,
)

data = load_workshop_data()
mri = data["mri"]
brain_outline = data["brain_mask"]
baseline_pet = data["baseline_pet"]
stress_pet = data["stress_pattern_pet"]
challenge_pet = data["challenge_pet"]
region_masks = {
    "Hippocampus": data["hippocampus_mask"],
    "Amygdala": data["amygdala_mask"],
    "Insula": data["insula_mask"],
}
print(f"Python {platform.python_version()} | NumPy {np.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Dataset: {mri.shape} | TemplateFlow {data['metadata']['templateflow_version']}")
print(f"Template: {data['metadata']['template']} | Atlas: {data['metadata']['atlas']}")
print("✓ Pre-flight imports and data load succeeded.")

## Suggested run sheet

| Time | Student section | Speaking prompt |
|---:|---|---|
| 0–7 | Welcome + modalities | Structure, glucose use, and BOLD answer different questions |
| 7–14 | Meet the regions | No single stress centre; networks and context matter |
| 14–27 | Acquire + manually register | A volunteer cannot be positioned identically in two scanners |
| 27–34 | Match score | A computer needs a numerical definition of better |
| 34–42 | Optimiser | Coarse-to-fine translation search; what changes if rotation is allowed? |
| 42–53 | Atlas measurement | Wrong alignment means wrong anatomical pixels |
| 53–60 | Compare patterns + recap | A simulation is not a diagnosis or universal effect |
| Optional 15 | fMRI bonus | Repetition converts a noisy time series into an activation map |

The core remains workable as a one-hour session. The bonus is suitable for extra time or independent follow-up.

## Science framing

- Call the PET images **simulated FDG patterns**, not patient scans.
- FDG-PET integrates tracer distribution over the uptake period; it is not an instantaneous stress meter.
- Stress does not have one universal regional pattern. Findings depend on task, timing, population, and analysis.
- The atlas regions are anatomically meaningful, but their brief functional descriptions are deliberately simplified.
- BOLD fMRI is an indirect haemodynamic signal. The bonus activation map is a correlation demonstration, not a full general linear model.
- Registration is one preprocessing step. Real pipelines may also include quality control, motion correction, noise reduction, scanner-specific corrections, and spatial normalisation.

## Atlas check

In [ ]:
show_region_atlas(data["region_mri"], data["region_masks"], data["metadata"]["regions"]);

## Answer key — registration

The best-scoring correction is **x = −7.2 mm, y = +5.3 mm**. Positive x moves image content right; positive y moves it down. The source grid is 1 mm, and interpolation allows sub-millimetre corrections.

In [ ]:
initial_score = how_well_matched(mri, challenge_pet)
manual_answer = shift_image(challenge_pet, x_offset_mm=-7.2, y_offset_mm=5.3)
manual_score = how_well_matched(mri, manual_answer)
automatic_pet, best_x, best_y, automatic_score = optimise(mri, challenge_pet, search_range=10)
print(f"Initial correlation: {initial_score:.4f} → display {normalise_match_score(initial_score):.3f} / 1")
print(f"Manual correlation:  {manual_score:.4f} → display {normalise_match_score(manual_score):.3f} / 1")
print(f"Automatic result:    {automatic_score:.4f} at x={best_x:+.1f} mm, y={best_y:+.1f} mm")
np.testing.assert_allclose((best_x, best_y), (-7.2, 5.3), atol=0.05)
assert automatic_score > initial_score
show_alignment(mri, automatic_pet, brain_outline, title="Expected automatic result");

The displayed score maps the useful correlation range for this exercise (`0.60` to `0.755`) onto `0` to `1`. Call it a **match score**, never percentage accuracy. Pearson correlation is kept intentionally transparent here; clinical multimodal registration often uses metrics designed for differing image contrasts.

## Answer key — atlas measurements

In [ ]:
figure, registered_values = show_region_measurements(
    automatic_pet, region_masks, title="Expected registered measurements"
)
wrong_amygdala = measure_region(challenge_pet, region_masks["Amygdala"])
print(f"Amygdala before registration: {wrong_amygdala:.3f}")
print(f"Amygdala after registration:  {registered_values['Amygdala']:.3f}")
assert registered_values["Amygdala"] > wrong_amygdala
comparison, baseline_values, stress_values = compare_pet_patterns(
    baseline_pet, stress_pet, region_masks
)
for name in region_masks:
    assert stress_values[name] > baseline_values[name]

The inserted regional increments are workshop design choices: amygdala +0.36, insula +0.24, and hippocampus +0.12 before PET blurring. They are not literature-derived effect sizes. The purpose is to make registration and atlas-based measurement visible.

## Bonus answer key — synthetic fMRI

In [ ]:
calculated_activation = calculate_correlation_map(
    data["fmri_timeseries"], data["fmri_task_design"]
)
np.testing.assert_allclose(calculated_activation, data["fmri_activation_map"], atol=1e-5)
injected = data["fmri_activation_mask"] > 0.30
brain = data["fmri_mean"] > 0.10
print(f"Mean correlation inside injected region: {calculated_activation[injected].mean():.3f}")
print(f"Mean elsewhere in brain:                 {calculated_activation[brain & ~injected].mean():.3f}")
show_fmri_result(
    data["fmri_mean"], data["fmri_task_blocks"], data["fmri_task_design"],
    data["fmri_roi_signal"], calculated_activation, threshold=0.35,
);

## Troubleshooting

- **A student gets lost:** Kernel → Restart Kernel and Run All, then return to the two-number cell. Its initial zero values are valid.
- **Data file missing locally:** run `python prepare_templateflow_data.py` once. Binder runs this during image build.
- **Plots appear twice:** keep the semicolon at the end of plotting cells.
- **Animation is static:** allow JavaScript for the notebook output, or skip directly to the exhaustive optimiser cell.
- **Internet unavailable in class:** launch Binder instances beforehand or prepare the data locally. Notebook execution itself makes no network requests.
- **Discussion runs short:** ask why a strong image correlation does not prove clinical validity, or what motion would do to an fMRI time series.

See `DATA_SOURCES.md` for exact resources, licences, processing, and scientific references.